In [ ]:
import pandas as pd
df = pd.read_csv('for_export_philippine_elections.csv')

authors = pd.read_csv('well_known_authors_philippine_elections.csv')

words = ['sb19', 'stell', 'jhoaiah', 'blooms', 'colaiah', 'barriofiestadxbjmfyang', 'jmfyang', 'bini', 'bebeBerries', 'jl',
                   'congrats', 'dustin', '#savewithmaya', 'housemate']

df = df[~df['text'].str.contains('|'.join(words), case=False, na=False)]

df.head()

,pseudo_id,text,retweetCount,replyCount,likeCount,quoteCount,viewCount,bookmarkCount,createdAt,lang,isReply,pseudo_conversationId,pseudo_inReplyToUsername,pseudo_author_userName,author_isBlueVerified
0,224179509746025,not a fan of Brent pero ano ba 'yang Big Winne...,0,0,0,0,5,0,2025-05-31 16:22:41+00:00,tl,False,224179509746025,NaN,733727272962306,False
1,998635197269390,wala talagang aasahan sa mga botante JUSKOPO,0,0,0,0,7,0,2025-05-31 16:21:56+00:00,tl,False,998635197269390,NaN,924044017517801,False
2,954085072967672,@975452594351818 Magiging presidente para guma...,0,0,0,0,0,0,2025-05-31 16:21:03+00:00,tl,True,725047262887831,9.754526e+14,685540093851332,False
3,90429221872722,Bumoto kahit walang idea kung ano to. Promise ...,0,0,0,0,2,0,2025-05-31 16:20:20+00:00,tl,False,90429221872722,NaN,897590766311451,False
4,503936424634617,alam mo talagang wikla duo ang bet ng CVs kase...,0,0,0,0,1,0,2025-05-31 16:19:10+00:00,tl,False,503936424634617,NaN,887555520780376,False


In [55]:
edges = df.groupby(['pseudo_author_userName','pseudo_inReplyToUsername']).size().reset_index(name='weight')
edges.sort_values('weight',ascending=False).head()
print(edges)

       pseudo_author_userName  pseudo_inReplyToUsername  weight
0                 74649779260              2.374105e+14       1
1                 74649779260              3.250027e+14       1
2                 74649779260              3.256803e+14       1
3                 74649779260              3.453692e+14       1
4                 74649779260              4.037803e+14       1
...                       ...                       ...     ...
58106         999889353237462              4.295322e+14       1
58107         999889353237462              9.551294e+14       1
58108         999896816974989              2.440071e+14       1
58109         999896816974989              9.998968e+14       2
58110         999909086562989              7.307986e+14       1

[58111 rows x 3 columns]


In [56]:
author_usernames = dict(zip(
    authors['obfuscated_userName'].astype(str).str[1:], 
    authors['author_userName']
))
print(author_usernames)

for i in range(len(edges)):
    id = str(int(edges.iloc[i]['pseudo_author_userName']))
    id2 = str(int(edges.iloc[i]['pseudo_inReplyToUsername']))
    if id in author_usernames:
        edges.at[i, 'pseudo_author_userName'] = author_usernames[id]
    if id2 in author_usernames:
        edges.at[i, 'pseudo_inReplyToUsername'] = author_usernames[id2] 

{'447129996589185': '__nisanjo', '502848624539866': '_djchacha', '894831624795349': '_GaiaPoly', '344694405381762': 'ABSCBN', '403780272519270': 'ABSCBNNews', '34859533045326': 'ABSCBNpr', '456793039343990': 'AFP', '876575320724228': 'AHOF__GRANDMA', '347280075511207': 'akazukinchan2x', '11503056031366': 'akoposimarcelo', '526730790366201': 'AltStarMagic', '467161425997003': 'alvinelchico', '802585864361733': 'amaxtriton', '208287835608527': 'ANCALERTS', '936960220706872': 'apangilinan', '70047054001669': 'Arshiethromycin', '712192195397021': 'AskPerplexity', '575306104619656': 'ateneodemanilau', '328126839578840': 'attyharryroque', '894416324110752': 'AttyLeiladeLima', '730798648352396': 'AttyMatula', '951392445303700': 'bamaquino', '768795647400867': 'BangkoSentral', '33643637499439': 'barrygutierrez3', '326683475544033': 'bayanmo', '942501146785377': 'BbMaharlika', '979339868045075': 'beabinene', '69469140131426': 'bgyo_ph', '607504780903547': 'BiniJhoannaOFC', '722855629086973': 'b

In [57]:
%matplotlib inline
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import warnings
from collections import Counter
import sys
import os
from __future__ import division
plt.rcParams["figure.figsize"] = (20,10)
import tqdm as tqdm

warnings.filterwarnings('ignore')
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:80% !important; }</style>"))

G = nx.from_pandas_edgelist(edges, 
                            source='pseudo_author_userName',
                            target='pseudo_inReplyToUsername',
                            edge_attr='weight',
                            create_using=nx.DiGraph())
print(G)
in_degrees = sorted(G.in_degree(), key=lambda x: x[1], reverse=True)
out_degrees = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)

# top 10 users
print("All in-degrees:", in_degrees[:10])
print("All out-degrees:", out_degrees[:10])   

DiGraph with 35772 nodes and 58111 edges
All in-degrees: [('ABSCBNNews', 1805), ('inquirerdotnet', 832), ('News5PH', 733), (623361697158128.0, 560), (901946442613399.0, 555), (776109641341820.0, 544), (36502604678147.0, 516), ('PhilippineStar', 499), ('newswatchplusph', 476), (556776310795431.0, 476)]
All out-degrees: [(439483351890134.0, 111), (500445945093556.0, 97), (602057431449805.0, 85), (245705656757626.0, 84), (925279596835042.0, 81), (88987228372981.0, 81), (554423716436549.0, 80), (810704260922273.0, 79), ('mayaofficialph', 75), (453836684166219.0, 75)]


In [58]:
print("All in-degrees:", in_degrees)
print("All out-degrees:", out_degrees)

All in-degrees: [('ABSCBNNews', 1805), ('inquirerdotnet', 832), ('News5PH', 733), (623361697158128.0, 560), (901946442613399.0, 555), (776109641341820.0, 544), (36502604678147.0, 516), ('PhilippineStar', 499), ('newswatchplusph', 476), (556776310795431.0, 476), (995120649715525.0, 469), ('smninews', 469), ('RichHeydarian', 430), (975404515899393.0, 430), (17206896528823.0, 378), ('gmanews', 322), (79008473933029.0, 298), (742776083943287.0, 282), ('LukeEspirituPH', 275), ('rapplerdotcom', 272), ('bamaquino', 263), (387006647385904.0, 259), (505173227680197.0, 259), ('PBBabscbn', 255), (141771773603661.0, 222), (697216793328325.0, 218), (664138562411848.0, 215), (587875972073636.0, 203), (106490152796335.0, 196), (388043756860081.0, 194), (304193516099150.0, 188), (487904401693285.0, 182), ('lilyganados', 169), (549139621537516.0, 167), (7650979450203.0, 166), (46329238896864.0, 164), (617268793518215.0, 163), ('PhilstarNews', 160), (943351515797164.0, 158), ('rowena_guanzon', 156), (86

In [59]:
communities = nx.community.louvain_communities(G, resolution=1, threshold=1e-7)

In [60]:
sizes = {}
for c in communities:
    length = len(c)
    sizes[length] = sizes.get(length, 0) + 1
count_comm = 0
for size, count in sorted(sizes.items(), reverse=True):
    print(f"Size {size}: {count}")
    count_comm += count
print(count_comm)

Size 3692: 1
Size 3458: 1
Size 3200: 1
Size 2539: 1
Size 1585: 1
Size 587: 1
Size 583: 1
Size 531: 1
Size 492: 1
Size 468: 1
Size 406: 1
Size 306: 1
Size 300: 1
Size 287: 1
Size 257: 1
Size 250: 1
Size 220: 1
Size 202: 1
Size 186: 1
Size 164: 1
Size 162: 1
Size 160: 1
Size 159: 1
Size 145: 1
Size 140: 1
Size 136: 1
Size 129: 1
Size 124: 1
Size 117: 1
Size 108: 1
Size 75: 1
Size 64: 1
Size 53: 1
Size 44: 1
Size 39: 1
Size 35: 1
Size 33: 1
Size 32: 1
Size 31: 1
Size 26: 2
Size 25: 1
Size 24: 1
Size 22: 2
Size 20: 1
Size 17: 1
Size 16: 3
Size 15: 4
Size 14: 4
Size 13: 2
Size 12: 4
Size 11: 6
Size 10: 7
Size 9: 15
Size 8: 9
Size 7: 17
Size 6: 41
Size 5: 69
Size 4: 180
Size 3: 500
Size 2: 3300
Size 1: 3980
8188


In [61]:
# results in betweenness_centrality.txt
# will take long to run
centrality = nx.betweenness_centrality(G)

In [62]:
cen = sorted(centrality.items(), key=lambda x: x[1], reverse=True)
for i in cen:
    print(i)

(94169936503831.0, 0.0034962072486950267)
(439483351890134.0, 0.0029094019787485772)
(226179221458297.0, 0.002201748311426849)
(975404515899393.0, 0.0019508739663201605)
(7595767113704, 0.001828534347326279)
(426098887240189.0, 0.0016089125792314498)
(556776310795431.0, 0.0015565817921709634)
(776109641341820.0, 0.0014903411376477602)
(453836684166219.0, 0.0013506611971914346)
(901946442613399.0, 0.0013188551333264901)
(243071933649732.0, 0.0012285402602726747)
(17206896528823.0, 0.0011851721782215278)
(387006647385904.0, 0.0011447436964123923)
(438468324738439.0, 0.001076722111508712)
(627382129095100.0, 0.0010561792095507723)
(125610889293945.0, 0.0010380026035909102)
('lilyganados', 0.0010103700777224411)
(941597642594992.0, 0.001009484466794749)
(36502604678147.0, 0.0009950441804998619)
(707866047743325.0, 0.0009754684763298298)
(298710194971273.0, 0.0009290892546018351)
(710497371297726.0, 0.0009206471389128502)
(141771773603661.0, 0.0009122466789633669)
(625446669809613.0, 0.0009

In [63]:
deg_centrality = nx.degree_centrality(G)
deg_cen = sorted(deg_centrality.items(), key=lambda x: x[1], reverse=True)
for i in deg_cen:
    print(i)

('ABSCBNNews', 0.05048782533337061)
('inquirerdotnet', 0.023287020211903497)
('News5PH', 0.020519415168712085)
(901946442613399.0, 0.015878784490229517)
(623361697158128.0, 0.015766962064241986)
(776109641341820.0, 0.015599228425260686)
(36502604678147.0, 0.014620782197869784)
('PhilippineStar', 0.013977803248441476)
(556776310795431.0, 0.013502557937994465)
(975404515899393.0, 0.01336277990551005)
('newswatchplusph', 0.013334824299013167)
('smninews', 0.013139135053534987)
(995120649715525.0, 0.013111179447038103)
('RichHeydarian', 0.012048866400156552)
(17206896528823.0, 0.010818819714293703)
('gmanews', 0.009029660898493193)
(79008473933029.0, 0.008414637555561768)
(742776083943287.0, 0.007883481032120993)
('LukeEspirituPH', 0.007715747393139695)
('rapplerdotcom', 0.0076318805736490455)
(387006647385904.0, 0.007492102541164631)
('bamaquino', 0.007352324508680216)
(505173227680197.0, 0.0072405020826926845)
('PBBabscbn', 0.007128679656705153)
(141771773603661.0, 0.006457745100779962)


In [64]:
deg_ave = 0.0
print(G.degree())
for i in G.degree():
    deg_ave += i[1]
deg_ave /= len(G.degree())
print("Average Degree: ", deg_ave)

[(74649779260, 12), (237410519977083.0, 54), ('newswatchplusph', 477), ('PhilippineStar', 500), (345369242965782.0, 2), ('ABSCBNNews', 1806), ('News5PH', 734), (654160991975367.0, 23), (864690867426781.0, 21), ('hecklerforever8', 68), (103725648483, 5), (64899621260454.0, 36), (506071831966741.0, 1), (527451248414886.0, 4), (620262767286476.0, 1), (917143843738134.0, 3), (155914800478, 1), (974987706257566.0, 2), (167559896847, 1), (169159415188, 1), (67279183811521.0, 5), (189184721447, 1), (217343838690896.0, 1), (248243993679, 1), (196503641684186.0, 1), (252645276986, 1), (570185673454027.0, 3), (297226483873, 1), (575949626431588.0, 9), (305144587363, 3), ('LukeEspirituPH', 276), (947807339984737.0, 20), (314045301596, 3), (170219666646542.0, 61), (563483373361007.0, 135), ('FifthSolomon', 28), (344521896144, 9), (412148754945, 1), (550171505045, 1), (295563215689978.0, 5), (560977799025, 2), (598934618469, 4), (138571055718112.0, 64), (482439483921493.0, 21), (612606680256255.0, 

In [65]:
for group in communities:
    if len(group) >= 1000:
        print(group)

{297455054544898, 373079888199683, 11125368586242, 875210874322947, 135786631634952.0, 998058697031691, 672640426500108, 898269720944654, 582177522262037, 337182783864853, 414150616514585, 124158043553818, 227123947544602, 158038007562271, 572389450276896, 854462498775074.0, 634977590517796, 675899603517477, 707375967297574.0, 861725963116582, 311052722135081, 323534666686506, 83697512431660.0, 170873340665901, 739355039768626, 'smninews', 491433245974584, 155223873839160, 300216467071034.0, 892027064598593, 801321343131714, 398900485734470, 984732660195408, 56799848259666, 765938778996819, 991393638031444, 892764741206099, 837616626065494, 684797165731927.0, 995467282014295, 415798961463385, 342577464123485, 934256182837341.0, 272846828421213.0, 942705622548580, 990878963867753.0, 305928345108587, 160489455673453, 988709709963374, 82533938954353, 850353305796724.0, 898874145882230, 807034374545527, 190523803893887, 243307635966080, 908241387937922, 387786445725826, 944057819504772, 56